This notebook contains cleaning, normalizing of GEE data and merging GEE with GSO data.

In [25]:
import pandas as pd
import numpy as np
from pathlib import Path

In [24]:
GEE_CSV = Path("../data/gee_features_raw.csv")
GSO_CSV = Path("../data/cleaned_gso.csv")

In [11]:
def clean_province_names(series):
    """Standardizes province string formatting."""
    mapping = {
        "Hai Phong City": "Hai Phong",
        "hai_phong_city": "Hai Phong",
    }
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", "_", regex=True)
        .replace(mapping)
    )

def clean_seasons(series):
    """Maps English/raw seasonal strings to uniform short codes."""
    season_map = {
        "spring": "dongxuan",
        "winter-spring": "dongxuan",
        "dongxuan": "dongxuan",
        "autumn": "hethu",
        "summer-autumn": "hethu",
        "hethu": "hethu",
        "winter": "thudong",
        "autumn-winter": "thudong",
        "thudong": "thudong"
    }
    return series.astype(str).str.lower().str.strip().replace(season_map)

def build_composite_key(df):
    """Creates a standardized composite key: province_year_season."""
    prov_str = clean_province_names(df["province_name"])
    year_str = df["year"].astype(str).str.strip()
    season_str = clean_seasons(df["season"])
    
    df["composite_key"] = prov_str + "_" + year_str + "_" + season_str
    return df

In [12]:
# 1. Load GEE Features
df_gee = pd.read_csv(GEE_CSV)
df_gee["year"] = df_gee["year"].astype(int)
df_gee = build_composite_key(df_gee)

# 2. Load GSO Statistics
df_gso = pd.read_csv(GSO_CSV)
df_gso["year"] = df_gso["year"].astype(int)
df_gso = build_composite_key(df_gso)

# Quick check on key alignment
print("Unique GEE keys:", df_gee["composite_key"].nunique())
print("Unique GSO keys:", df_gso["composite_key"].nunique())

Unique GEE keys: 4032
Unique GSO keys: 3117


In [13]:
missing_in_gso = set(df_gee["composite_key"]) - set(df_gso["composite_key"])
missing_in_gee = set(df_gso["composite_key"]) - set(df_gee["composite_key"])

print(f"GEE keys missing in GSO: {len(missing_in_gso)}")
print(f"GSO keys missing in GEE: {len(missing_in_gee)}")

if len(missing_in_gso) > 0:
    print("Sample missing in GSO:", list(missing_in_gso)[:5])

GEE keys missing in GSO: 1146
GSO keys missing in GEE: 231
Sample missing in GSO: ['dien_bien_2011_hethu', 'son_la_2010_hethu', 'thanh_hoa_2023_hethu', 'ca_mau_2014_dongxuan', 'thai_nguyen_2013_hethu']


In [15]:
# Check unique seasons per province in GSO vs GEE
print("GSO unique seasons:", df_gso["season"].unique())
print("GEE unique seasons:", df_gee["season"].unique())

# Check a specific missing province like Dien Bien in GSO
print(df_gso[df_gso["province_name"].str.contains("dien_bien", na=False)][["year", "season"]])

GSO unique seasons: <StringArray>
['autumn', 'spring', 'winter']
Length: 3, dtype: str
GEE unique seasons: <StringArray>
['thudong', 'dongxuan', 'hethu']
Length: 3, dtype: str
Empty DataFrame
Columns: [year, season]
Index: []


In [17]:
import unicodedata

def remove_diacritics(text):
    """Removes Vietnamese diacritics for robust matching."""
    nfkd_form = unicodedata.normalize('NFKD', str(text))
    return "".join([c for c in nfkd_form if not unicodedata.combining(c)])

# Inspect raw GSO rows for Dien Bien by stripping diacritics first
raw_gso = pd.read_csv(GSO_CSV)
raw_gso["search_name"] = raw_gso["province_name"].apply(remove_diacritics).str.lower()

dien_bien_rows = raw_gso[raw_gso["search_name"].str.contains("dien bien", na=False)]
print(dien_bien_rows[["province_name", "year", "season"]])

    province_name  year  season
871     Dien Bien  2005  spring
872     Dien Bien  2005  winter
873     Dien Bien  2006  spring
874     Dien Bien  2006  winter
875     Dien Bien  2007  spring
876     Dien Bien  2007  winter
877     Dien Bien  2008  spring
878     Dien Bien  2008  winter
879     Dien Bien  2009  spring
880     Dien Bien  2009  winter
881     Dien Bien  2010  spring
882     Dien Bien  2010  winter
883     Dien Bien  2011  spring
884     Dien Bien  2011  winter
885     Dien Bien  2012  spring
886     Dien Bien  2012  winter
887     Dien Bien  2013  spring
888     Dien Bien  2013  winter
889     Dien Bien  2014  spring
890     Dien Bien  2014  winter
891     Dien Bien  2015  spring
892     Dien Bien  2015  winter
893     Dien Bien  2016  spring
894     Dien Bien  2016  winter
895     Dien Bien  2017  spring
896     Dien Bien  2017  winter
897     Dien Bien  2018  spring
898     Dien Bien  2018  winter
899     Dien Bien  2019  spring
900     Dien Bien  2019  winter
901     

Dien Bien, the GSO data only records spring and winter (no autumn / hethu).

In [23]:
# Inner merge on composite key
merged = df_gee.merge(
    df_gso,
    on="composite_key",
    how="inner",
    suffixes=("_gee", "_gso")
)

# Clean up duplicate key columns post-merge
for col in ["province_name", "year", "season"]:
    if f"{col}_gso" in merged.columns:
        merged[col] = merged[f"{col}_gso"]
        merged = merged.drop(columns=[f"{col}_gee", f"{col}_gso"])


# Bypass the duplicate registration error if it occurs
try:
    _period_type = pyarrow.lib.ExtensionType("pandas.period")
except Exception:
    pass

OUTPUT_CSV = Path("../data/vietnam_rice_dataset.csv")
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
merged.to_csv(OUTPUT_CSV, index=False)

print(f"Successfully merged! Final dataset shape: {merged.shape}")
print(f"Saved to: {OUTPUT_CSV}")

Successfully merged! Final dataset shape: (2886, 29)
Saved to: ..\data\vietnam_rice_dataset.csv
